# 02. 실험 실행 창구 (Colab T4 / Kaggle 백업, ADR-0005)

역할: **브랜치 체크아웃 -> 데이터 준비 -> `scripts/run_experiment.py` 실행 -> 실행 기록 커밋/push**

- 로직은 모두 `src/sr_pipeline/`에 두며, 이 노트북에는 학습·평가 로직을 작성하지 않습니다.
- 실행 기록(`results/runs/<run_id>/`)은 **본인 이름 브랜치**에 커밋합니다. main에는 직접 push하지 않습니다(PR + 승인 1명).
- Colab과 Kaggle은 같은 명령을 쓰고 **경로 인자만** 다릅니다(셀 1이 환경을 감지해 채웁니다).
  - Colab: 공유 Drive의 압축 파일 1개를 런타임 로컬로 복사해 `--data-archive`로 풀어 씁니다. Drive 마운트가 안 되면 셀 3-1(공유 링크 `gdown`)로 받습니다.
  - Kaggle: 비공개 Dataset이 `/kaggle/input/...`에 붙으므로 `--data-dir`로 그 폴더를 그대로 씁니다.
- `configs/fair_comparison.yaml`의 `pilot_determined`가 `null`인 동안(T4.2 전)에는 본 실행이 시작 전에 거부됩니다. 이는 정상입니다.
- **토큰은 셀 6에서 `getpass`로만 입력**합니다. 파일·출력·`.git/config`에 남지 않습니다. 실행 후 이 노트북의 셀 출력은 지우고 커밋하세요.

In [ ]:
# 1. 실행 파라미터 (환경을 감지해 경로를 채운다)
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
BRANCH = "YongMin"        # 본인 이름 브랜치 (YongMin / Gawoun). main은 push하지 않는다.
CONFIG_ID = "baseline"    # bicubic, baseline, A, B, AB, C, Bp (+ 승격 시 AC, BC, ABC)
SEED = 0                  # 0, 1, 2

IN_KAGGLE = Path("/kaggle/input").exists()
if IN_KAGGLE:
    DATA_DIR = "/kaggle/input"                      # 붙인 Dataset 아래에서 patch_pool/을 찾는다(읽기 전용, 복사 없음). 패치 풀이 든 Dataset을 둘 이상 붙였다면 그 폴더로 좁힌다.
    DATA_ARCHIVE = None
    CKPT_DIR = "/kaggle/working/checkpoints"        # Kaggle에는 Drive가 없다. 필요하면 working 출력을 내려받는다.
    WORK = Path("/kaggle/working")
else:
    DATA_DIR = "/content/data"                      # 런타임 로컬 해제 위치
    DATA_ARCHIVE = "/content/sr_data.zip"           # Drive에서 복사한(또는 gdown으로 받은) 압축 파일
    CKPT_DIR = "/content/drive/MyDrive/vp_project/checkpoints"   # Drive를 못 쓰면 "/content/checkpoints" + 셀 6의 COMMIT_WEIGHTS=True
    WORK = Path("/content")
DRIVE_ARCHIVE = "/content/drive/MyDrive/vp_project/sr_data.zip"  # 공유 Drive의 압축 파일 위치(폴더 구성에 맞게 수정)
DATA_SOURCE = "drive"     # Colab: "drive"(Drive 마운트 후 복사) 또는 "gdown"(마운트가 안 될 때, 공유 링크로 받기 — ADR-0005 대체 경로)
ARCHIVE_SHA256 = ""       # 원본 zip의 sha256(64자리). 주면 해제 전에 확인한다(특히 gdown으로 받았을 때 권장).
REPO_DIR = WORK / "VideoProcessing-Project"
assert BRANCH != "main", "실행 기록은 이름 브랜치에 커밋합니다."
print("환경:", "Kaggle" if IN_KAGGLE else "Colab/로컬", "| 브랜치:", BRANCH, "| 구성:", CONFIG_ID, "| 시드:", SEED)

In [ ]:
# 2. 브랜치 가져오기
if (REPO_DIR / ".git").is_dir():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!git log --oneline -3

In [ ]:
# 3. 데이터 준비
# Colab: 압축 파일 1개만 런타임 로컬로 가져온다(작은 파일 수만 개를 Drive에서 복사하면 매우 느리다). 해제는 셀 4의 실행 명령이 한다.
#   DATA_SOURCE = "drive": Drive를 마운트해 복사.   "gdown": 마운트 없이 공유 링크로 받기(ADR-0005 대체 경로; 링크는 getpass로만 입력, 출력·파일에 남기지 않음).
# Kaggle: 할 일 없음(Dataset이 /kaggle/input에 붙어 있어야 한다. 오른쪽 Input 패널의 Add Input).
if not IN_KAGGLE:
    if DATA_SOURCE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        !cp "{DRIVE_ARCHIVE}" "{DATA_ARCHIVE}"
    elif DATA_SOURCE == "gdown":
        import getpass, subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
        link = getpass.getpass("Drive 공유 링크(zip 파일 하나, 화면에 표시되지 않음): ")
        try:
            gdown.download(url=link, output=DATA_ARCHIVE, fuzzy=True, quiet=True)
        finally:
            del link
        assert Path(DATA_ARCHIVE).is_file(), "내려받기 실패: 링크 권한(링크가 있는 모든 사용자)과 파일 링크 여부를 확인하세요."
    else:
        raise ValueError(f"DATA_SOURCE는 'drive' 또는 'gdown'이어야 합니다: {DATA_SOURCE!r}")
    !ls -l "{DATA_ARCHIVE}"

In [ ]:
# 4. 단일 실행 (학습 + 통합 평가 + 실행 기록 저장). Colab과 Kaggle이 같은 명령이고 경로 인자만 다르다.
ARCHIVE_ARG = f"--data-archive {DATA_ARCHIVE}" if DATA_ARCHIVE else ""
if DATA_ARCHIVE and ARCHIVE_SHA256:
    ARCHIVE_ARG += f" --archive-sha256 {ARCHIVE_SHA256}"
RUNS_ROOT = REPO_DIR / "results" / "runs"
RECORDS_BEFORE = {p.name for p in RUNS_ROOT.glob("*")} if RUNS_ROOT.exists() else set()   # 이번 실행이 새로 만드는 폴더를 찾으려고
!PYTHONPATH=src python scripts/run_experiment.py --config {CONFIG_ID} --seed {SEED} --data-dir {DATA_DIR} {ARCHIVE_ARG} --checkpoint-dir {CKPT_DIR}
# 중간에 런타임이 끊겼다면 같은 명령에 --resume-run-id <run_id> 를 붙여 이어서 학습한다(코드 커밋이 같을 때만 가능).

In [ ]:
# 5. 실행 기록 확인
new_records = sorted({p.name for p in RUNS_ROOT.glob("*")} - RECORDS_BEFORE)
assert len(new_records) == 1, f"이번 실행이 새로 만든 실행 기록 폴더가 {len(new_records)}개입니다(1개여야 함): {new_records}"
RUN_DIR = RUNS_ROOT / new_records[0]
RUN_ID = RUN_DIR.name
print("run_id:", RUN_ID)
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(RUN_DIR), f"{path.stat().st_size:,} B")
import pandas as pd
display(pd.read_csv(RUN_DIR / "summary.csv"))

In [ ]:
# 6. 실행 기록 커밋 + push (이름 브랜치). 토큰은 getpass로만 받고 어디에도 저장하지 않는다.
# - 커밋은 results/runs/<run_id>/ 만 담는다.
# - push가 실패하면 커밋은 로컬에 남고, 실행 기록 zip을 내려받아 로컬 VS Code에서 내 Git 자격 증명으로 커밋한다(ADR-0005).
import getpass
sys.path.insert(0, str(REPO_DIR / "src"))
from sr_pipeline.engine.publish import PushFailed, bundle_run_record, commit_run_record, push_branch

COMMIT_WEIGHTS = False   # Drive를 쓸 수 없는 경우에만 True: 평가 체크포인트(weights.pt, 약 150KB)를 실행 기록과 함께 커밋한다.
GH_USER = input("GitHub 사용자명: ").strip()
GIT_NAME = input("커밋 작성자 이름 (이 커밋에만 사용): ").strip()
GIT_EMAIL = input("커밋 작성자 이메일 (이 커밋에만 사용): ").strip()
sha = commit_run_record(
    REPO_DIR, RUN_DIR, message=f"results: {CONFIG_ID} seed {SEED}", author_name=GIT_NAME, author_email=GIT_EMAIL,
    weights_path=Path(CKPT_DIR) / RUN_ID / "weights.pt" if COMMIT_WEIGHTS else None,
    trained=CONFIG_ID != "bicubic",   # Bicubic 하한은 학습하지 않아 train_log.json이 없다
)
print("커밋:", sha[:8])
token = getpass.getpass("GitHub 토큰 (저장되지 않음): ")
try:
    push_branch(REPO_DIR, BRANCH, username=GH_USER, token=token)
    print("push 완료:", BRANCH)
except PushFailed as error:
    print(error)
    bundle = bundle_run_record(RUN_DIR, WORK / f"{RUN_ID}.zip")
    print("실행 기록 zip을 내려받아 로컬 VS Code에서 내 Git 자격 증명으로 커밋하세요(ADR-0005):", bundle)
    try:
        from google.colab import files
        files.download(str(bundle))      # Colab: 브라우저/확장으로 바로 내려받는다. Kaggle은 Output 탭에서 받는다.
    except ImportError:
        pass
finally:
    del token